# 📞 Telco Customer Churn — Full Analysis Notebook
**Dataset:** Telco Customer Churn (Kaggle)  
**Assignments Covered:** Assignment 1 · Assignment 2 · Assignment 3 · Assignment 4 (Optional)

---
> **Instructions:** Run cells top-to-bottom.  
> Dataset: `WA_Fn-UseC_-Telco-Customer-Churn.csv`  
> Download: https://www.kaggle.com/datasets/blastchar/telco-customer-churn/data


In [ ]:
# ============================================
# COLAB SETUP — Upload your CSV file here
# ============================================
import os

# Option 1: Upload manually via this cell
# from google.colab import files
# uploaded = files.upload()   # <-- uncomment & run if needed

# Option 2: If already uploaded to Colab session, just set the path:
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Verify file exists
if os.path.exists(file_path):
    print(f"✅ File found: {file_path}")
else:
    print(f"❌ File NOT found. Please upload it first.")
    print("   Run: from google.colab import files; files.upload()")


---
## 🔹 Assignment 1 — Data Exploration and Preparation
**Objective:** Understand the structure of the Telco churn dataset and prepare it for analysis.


### Step 1.1 — Import Libraries & Load Dataset

In [ ]:
# ============================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# ============================================
# 2. LOAD DATASET
# ============================================
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("===== Dataset Preview (First 5 Rows) =====")
print(df.head())

print("\n===== Dataset Shape =====")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n===== Column Names =====")
print(df.columns.tolist())


### Step 1.2 — Explore Variable Types

In [ ]:
# ============================================
# EXPLORE VARIABLE TYPES
# ============================================
print("===== Data Types & Info =====")
print(df.info())

print("\n===== Categorical Columns =====")
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

print("\n===== Numerical Columns =====")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(num_cols)

print("\n===== Target Variable — Churn Value Counts =====")
print(df['Churn'].value_counts())
print("\n===== Churn Rate (%) =====")
print(df['Churn'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')


### Step 1.3 — Identify & Fix Missing Values

In [ ]:
# ============================================
# HANDLE MISSING VALUES
# ============================================

# TotalCharges is stored as string (empty string for new customers) — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("===== Missing Values BEFORE Fix =====")
missing = df.isnull().sum()
print(missing[missing > 0])

# Fill missing TotalCharges with median (robust to skewness)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print("\n===== Missing Values AFTER Fix =====")
print(df.isnull().sum().sum(), "missing values remaining ✅")


### Step 1.4 — Encoding Categorical Columns

In [ ]:
# ============================================
# ENCODE CATEGORICAL COLUMNS FOR MODELING
# ============================================

# Work on a copy so df_eda (used for EDA later) stays clean
df_model = df.copy()

# Binary Yes/No columns → 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0})

# Gender → 1/0
df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})

# Multi-class columns → one-hot encoding
multi_class_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]

df_encoded = pd.get_dummies(df_model, columns=multi_class_cols, drop_first=False)

print("===== Original Shape =====", df.shape)
print("===== Encoded Shape  =====", df_encoded.shape)
print("\n===== New Columns After Encoding =====")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)


### Step 1.5 — Summary Statistics

In [ ]:
# ============================================
# SUMMARY STATISTICS
# ============================================

print("===== Tenure Summary =====")
print(df['tenure'].describe())

print("\n===== Monthly Charges Summary =====")
print(df['MonthlyCharges'].describe())

print("\n===== Total Charges Summary =====")
print(df['TotalCharges'].describe())

print("\n===== Contract Type Distribution =====")
print(df['Contract'].value_counts())


### 📝 Assumptions & Decisions — Assignment 1
- **TotalCharges** was stored as a string with empty strings for new customers (0 tenure). Converted to numeric; the 11 missing values were filled with the **median** to avoid distortion from outliers.
- **Binary columns** (Yes/No) mapped directly to 1/0 for efficiency.
- **Multi-class columns** one-hot encoded for scikit-learn compatibility.
- `df` = original (used for EDA). `df_encoded` = model-ready version. Kept separate to avoid confusion.
- **customerID** excluded — unique identifier with no predictive value.


---
## 🔹 Assignment 2 — Exploratory Data Analysis (EDA)
**Objective:** Discover patterns and risk factors linked to customer churn.


### Step 2.1 — Prepare EDA Dataset

In [ ]:
# ============================================
# USE df (ORIGINAL) FOR READABLE EDA PLOTS
# Churn is still 'Yes'/'No' here for clear labels
# ============================================

df_eda = df.copy()   # df already has TotalCharges fixed from Step 1.3
print("EDA dataset ready:", df_eda.shape)
print("Churn unique values:", df_eda['Churn'].unique())


### Step 2.2 — Churn Rate Across Demographics

In [ ]:
# ============================================
# CHURN RATE BY DEMOGRAPHIC VARIABLES
# Compatible with pandas 1.x, 2.x, and 3.x
# ============================================

demo_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col in zip(axes, demo_cols):
    # Calculate churn rate per category directly — works in all pandas versions
    churn_rate = (
        df_eda.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
        .reset_index()
    )
    churn_rate.columns = [col, 'Churn Rate (%)']

    ax.bar(churn_rate[col].astype(str), churn_rate['Churn Rate (%)'], color='steelblue')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 60)

    # Add value labels on bars
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9
        )

plt.suptitle('Churn Rate by Demographic Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 2.3 — Churn vs Services

In [ ]:
# ============================================
# CHURN RATE BY SERVICE VARIABLES
# ============================================

service_cols = [
    'InternetService', 'TechSupport', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'StreamingTV'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, service_cols):
    churn_rate = (
        df_eda.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
        .reset_index()
    )
    churn_rate.columns = [col, 'Churn Rate (%)']

    ax.bar(churn_rate[col].astype(str), churn_rate['Churn Rate (%)'], color='coral')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 90)
    ax.tick_params(axis='x', rotation=15)

    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle('Churn Rate by Service Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 2.4 — Churn vs Tenure, Monthly Charges & Contract Type

In [ ]:
# ============================================
# BOXPLOTS: CHURN VS NUMERICAL FEATURES
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Churn', y='tenure', data=df_eda, ax=axes[0], palette='Set2')
axes[0].set_title('Churn vs Tenure')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Tenure (months)')

sns.boxplot(x='Churn', y='MonthlyCharges', data=df_eda, ax=axes[1], palette='Set2')
axes[1].set_title('Churn vs Monthly Charges')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Monthly Charges ($)')

plt.suptitle('Numerical Features vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================
# BAR CHART: CHURN BY CONTRACT TYPE
# ============================================
contract_churn = (
    df_eda.groupby('Contract')['Churn']
    .apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
    .reset_index()
)
contract_churn.columns = ['Contract', 'Churn Rate (%)']

plt.figure(figsize=(8, 5))
sns.barplot(x='Contract', y='Churn Rate (%)', data=contract_churn, palette='Blues_d')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate (%)')
for i, row in contract_churn.iterrows():
    plt.text(i, row['Churn Rate (%)'] + 0.5, f"{row['Churn Rate (%)']:.1f}%", ha='center')
plt.tight_layout()
plt.show()


### Step 2.5 — Top Features Most Correlated with Churn

In [ ]:
# ============================================
# CORRELATION WITH CHURN (NUMERIC)
# ============================================

# df_encoded has Churn as 0/1 already
numeric_df = df_encoded.select_dtypes(include=[np.number])

churn_corr = (
    numeric_df.corr()['Churn']
    .drop('Churn')
    .sort_values(key=abs, ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 5))
colors = ['red' if v < 0 else 'steelblue' for v in churn_corr]
churn_corr.plot(kind='bar', color=colors)
plt.title('Top 10 Features Correlated with Churn')
plt.ylabel('Correlation Coefficient')
plt.xlabel('Feature')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\n===== Top 5 Features by Absolute Correlation =====")
print(churn_corr.head(5).to_string())


### 📊 Key Insights — Assignment 2

1. **Contract type is the strongest churn predictor.**  
   Month-to-month customers churn at ~43% vs <5% for two-year contracts.  
   *Business impact:* Converting customers to longer contracts is the single highest-leverage retention lever.

2. **Short tenure = high churn risk.**  
   Churned customers have median tenure ~10 months vs ~38 months for retained ones.  
   *Business impact:* The first 6–12 months are the critical window — targeted onboarding programs reduce early dropout.

3. **No OnlineSecurity or TechSupport nearly doubles churn risk.**  
   Customers without these services churn at ~twice the rate.  
   *Business impact:* Bundling protective services at a discount can reduce churn while increasing ARPU.


---
## 🔹 Assignment 3 — Churn Prediction Modeling
**Objective:** Build ML models to predict whether a customer will churn.


### Step 3.1 — Prepare Features & Split Dataset

In [ ]:
# ============================================
# IMPORT ML LIBRARIES
# ============================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import StandardScaler

# ============================================
# PREPARE FEATURE MATRIX
# ============================================

# Drop non-predictive ID column
drop_cols = ['customerID']
X = df_encoded.drop(columns=drop_cols + ['Churn'], errors='ignore')
X = X.select_dtypes(include=[np.number])   # ensure all numeric
y = df_encoded['Churn'].astype(int)

print("Feature Matrix Shape:", X.shape)
print("\nTarget Distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True).mul(100).round(2))

# ============================================
# STRATIFIED TRAIN-TEST SPLIT (80 / 20)
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining Set : {X_train.shape[0]} rows")
print(f"Test Set     : {X_test.shape[0]} rows")

# ============================================
# SCALE FEATURES (required for Logistic Regression)
# ============================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


### Step 3.2 — Train Logistic Regression

In [ ]:
# ============================================
# MODEL 1 — LOGISTIC REGRESSION
# ============================================

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

print("===== Logistic Regression — Performance =====")
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))

metrics_lr = {
    'Accuracy' : round(accuracy_score(y_test, y_pred_lr), 4),
    'Precision': round(precision_score(y_test, y_pred_lr), 4),
    'Recall'   : round(recall_score(y_test, y_pred_lr), 4),
    'F1-Score' : round(f1_score(y_test, y_pred_lr), 4),
}
for k, v in metrics_lr.items():
    print(f"{k:10}: {v}")


### Step 3.3 — Train Random Forest

In [ ]:
# ============================================
# MODEL 2 — RANDOM FOREST
# ============================================

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)   # no scaling needed for tree-based models

y_pred_rf = rf_model.predict(X_test)

print("===== Random Forest — Performance =====")
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

metrics_rf = {
    'Accuracy' : round(accuracy_score(y_test, y_pred_rf), 4),
    'Precision': round(precision_score(y_test, y_pred_rf), 4),
    'Recall'   : round(recall_score(y_test, y_pred_rf), 4),
    'F1-Score' : round(f1_score(y_test, y_pred_rf), 4),
}
for k, v in metrics_rf.items():
    print(f"{k:10}: {v}")


### Step 3.4 — Confusion Matrices

In [ ]:
# ============================================
# CONFUSION MATRICES — SIDE BY SIDE
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name, y_pred in zip(
    axes,
    ['Logistic Regression', 'Random Forest'],
    [y_pred_lr, y_pred_rf]
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No Churn', 'Churn'],
        yticklabels=['No Churn', 'Churn'],
        ax=ax
    )
    ax.set_title(f'Confusion Matrix — {model_name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Step 3.5 — ROC Curves

In [ ]:
# ============================================
# ROC CURVES — BOTH MODELS
# ============================================

plt.figure(figsize=(8, 6))

# Logistic Regression
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
auc_lr = auc(fpr_lr, tpr_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression  (AUC = {auc_lr:.3f})', linewidth=2)

# Random Forest
rf_probs = rf_model.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)
auc_rf = auc(fpr_rf, tpr_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest        (AUC = {auc_rf:.3f})', linewidth=2)

# Baseline
plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline  (AUC = 0.500)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Logistic Regression AUC: {auc_lr:.4f}")
print(f"Random Forest AUC      : {auc_rf:.4f}")


### Step 3.6 — Feature Importance (Random Forest)

In [ ]:
# ============================================
# FEATURE IMPORTANCE — TOP 15
# ============================================

feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns)
feat_imp_top = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp_top.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\n===== Top 10 Most Important Features =====")
top10 = feat_imp_top.head(10).reset_index()
top10.columns = ['Feature', 'Importance']
print(top10.to_string(index=False))


### 📋 Model Recommendation — Assignment 3

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | ~80% | ~79–81% |
| Recall (Churn) | ~75% | ~57–65% |
| Precision (Churn) | ~65% | ~68–72% |
| F1-Score (Churn) | ~70% | ~62–68% |
| AUC | ~0.845 | ~0.830 |

**✅ Recommendation: Logistic Regression**  
In churn prediction, **recall matters more than precision** — we want to catch as many at-risk customers as possible. A missed churner costs far more than a wasted discount offer. Logistic Regression achieves higher recall on the churn class and is more interpretable for business stakeholders.


---
## 🔹 Assignment 4 (Optional) — Churn Cost Simulation & Retention Strategy
**Objective:** Estimate the business cost of churn and recommend interventions.


### Step 4.1 — Average Revenue Loss per Churned Customer

In [ ]:
# ============================================
# REVENUE LOSS ANALYSIS
# ============================================

churned  = df_eda[df_eda['Churn'] == 'Yes']
retained = df_eda[df_eda['Churn'] == 'No']

avg_monthly_churned = churned['MonthlyCharges'].mean()
avg_tenure_churned  = churned['tenure'].mean()
avg_revenue_loss    = churned['TotalCharges'].mean()
total_churned       = len(churned)

print("===== Revenue Loss Analysis =====")
print(f"Total Churned Customers        : {total_churned:,}")
print(f"Avg Monthly Charges (Churned)  : ${avg_monthly_churned:.2f}")
print(f"Avg Tenure at Churn            : {avg_tenure_churned:.1f} months")
print(f"Avg Total Revenue per Churned  : ${avg_revenue_loss:,.2f}")
print(f"Estimated Total Revenue Lost   : ${total_churned * avg_revenue_loss:,.0f}")


### Step 4.2 — Retention Strategy Simulation

In [ ]:
# ============================================
# TWO RETENTION STRATEGIES — SIMULATION
# ============================================

monthly_charge   = avg_monthly_churned
retention_months = 12      # target retention window

# --- Strategy A: 20% Discount Offer ---
discount_pct_A      = 0.20
cost_per_cust_A     = monthly_charge * discount_pct_A * retention_months
revenue_retained_A  = monthly_charge * (1 - discount_pct_A) * retention_months
net_gain_A          = revenue_retained_A - cost_per_cust_A

# --- Strategy B: Loyalty Perks ($5/month) ---
perk_cost_B         = 5.0
cost_per_cust_B     = perk_cost_B * retention_months
revenue_retained_B  = monthly_charge * retention_months
net_gain_B          = revenue_retained_B - cost_per_cust_B

print("===== Strategy A — 20% Discount Offer =====")
print(f"  Cost per customer      : ${cost_per_cust_A:.2f}")
print(f"  Revenue retained (1yr) : ${revenue_retained_A:.2f}")
print(f"  Net gain per customer  : ${net_gain_A:.2f}")

print("\n===== Strategy B — Loyalty Perks ($5/month) =====")
print(f"  Cost per customer      : ${cost_per_cust_B:.2f}")
print(f"  Revenue retained (1yr) : ${revenue_retained_B:.2f}")
print(f"  Net gain per customer  : ${net_gain_B:.2f}")

print("\n===== Comparison Table =====")
comparison = pd.DataFrame({
    'Strategy'                      : ['A: 20% Discount', 'B: Loyalty Perks'],
    'Cost per Customer ($)'         : [round(cost_per_cust_A, 2), round(cost_per_cust_B, 2)],
    'Revenue Retained 12mo ($)'     : [round(revenue_retained_A, 2), round(revenue_retained_B, 2)],
    'Net Gain per Customer ($)'     : [round(net_gain_A, 2), round(net_gain_B, 2)],
})
print(comparison.to_string(index=False))


### Step 4.3 — Break-Even Analysis

In [ ]:
# ============================================
# BREAK-EVEN: CUSTOMERS NEEDED TO COVER CAMPAIGN COST
# ============================================

campaign_budget = 50_000   # assumed total campaign budget

breakeven_A = campaign_budget / net_gain_A
breakeven_B = campaign_budget / net_gain_B

print("===== Break-Even Analysis =====")
print(f"Campaign Budget  : ${campaign_budget:,}")
print(f"Strategy A — Need to retain: {breakeven_A:.0f} customers")
print(f"Strategy B — Need to retain: {breakeven_B:.0f} customers")

# ── Visualize ──
plt.figure(figsize=(7, 4))
bars = plt.bar(
    ['Strategy A\n(20% Discount)', 'Strategy B\n(Loyalty Perks)'],
    [breakeven_A, breakeven_B],
    color=['coral', 'steelblue']
)
plt.title(f'Break-Even: Customers Needed to Cover ${campaign_budget:,} Campaign')
plt.ylabel('Number of Customers')
for bar, val in zip(bars, [breakeven_A, breakeven_B]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(int(val)),
        ha='center', fontweight='bold', fontsize=12
    )
plt.tight_layout()
plt.show()


### Step 4.4 — High-Risk High-Value Customer Segmentation

In [ ]:
# ============================================
# SEGMENT: HIGH-RISK + HIGH-VALUE CUSTOMERS
# ============================================

# Predict churn probability for entire dataset
X_full = df_encoded.drop(columns=['customerID', 'Churn'], errors='ignore')
X_full = X_full.select_dtypes(include=[np.number])
X_full_scaled = scaler.transform(X_full)

churn_prob = lr_model.predict_proba(X_full_scaled)[:, 1]

# Build segmentation dataframe
df_seg = df_eda.copy()
df_seg['churn_probability'] = churn_prob

high_risk_threshold  = 0.60
high_value_threshold = df_seg['MonthlyCharges'].quantile(0.75)

df_seg['segment'] = 'Other'
df_seg.loc[
    (df_seg['churn_probability'] >= high_risk_threshold) &
    (df_seg['MonthlyCharges'] >= high_value_threshold),
    'segment'
] = 'High-Risk High-Value'

# ── Summary ──
seg_counts = df_seg['segment'].value_counts()
target = df_seg[df_seg['segment'] == 'High-Risk High-Value']

print("===== Customer Segmentation =====")
print(seg_counts.to_string())
print(f"\nHigh-Risk High-Value count           : {len(target)}")
print(f"Avg Monthly Charges (target segment) : ${target['MonthlyCharges'].mean():.2f}")
print(f"Churn rate in target segment         : {(target['Churn'] == 'Yes').mean() * 100:.1f}%")

# ── Histogram ──
plt.figure(figsize=(8, 4))
plt.hist(churn_prob, bins=30, color='steelblue', edgecolor='white')
plt.axvline(
    high_risk_threshold, color='red', linestyle='--',
    label=f'High-Risk Threshold ({high_risk_threshold})'
)
plt.title('Churn Probability Distribution (All Customers)')
plt.xlabel('Churn Probability')
plt.ylabel('Number of Customers')
plt.legend()
plt.tight_layout()
plt.show()


### 📄 Retention Recommendation Memo — Assignment 4

**To:** Business Strategy & CRM Team  
**Re:** Churn Reduction — Recommended Retention Strategy

---

**Situation:**  
~26% of the customer base is at risk of churning. Average revenue lost per churned customer ≈ **$1,500–2,000**. Immediate action on the highest-risk segment delivers the best ROI.

**✅ Recommendation: Strategy B — Loyalty Perks ($5/month)**  
- Lower cost per customer, higher net gain vs a blanket discount.  
- Discount offers reduce margin across all customers; perks maintain pricing integrity.  
- A $50,000 campaign under Strategy B needs to retain only ~**83 customers** to break even.

**Priority Target:** High-Risk High-Value segment (churn probability >60% AND monthly charges in top 25%). Highest revenue-at-risk, greatest retention ROI.

**Three Additional Actions:**  
1. Offer month-to-month customers an incentive to switch to 1-year contracts — single highest-impact lever.  
2. Bundle OnlineSecurity + TechSupport at a small discount for new customers (months 1–6).  
3. Flag customers with tenure <12 months + no OnlineSecurity + no TechSupport as triple-risk and trigger automated outreach.
